
# Phase 3C — Evaluation Protocol Comparison

**Experiment:** `phase_3c_evaluation_protocol_comparison`

This notebook resolves the Phase 3A vs Phase 3B duplicate-group-aware split discrepancy and compares candidate evaluation protocols without modifying production code or the canonical dataset.

### Decision set

- **A — Random row split:** reproduces the Phase 2 baseline and quantifies duplicate overlap.
- **B — Deterministic feature-group-aware split, all groups:** identical feature vectors are kept on one side of the split.
- **C — Deterministic feature-group-aware split, conflicting groups excluded:** sensitivity analysis only; no rows are deleted from the canonical dataset.
- **D — Derived-label policy:** documented but **not treated as an admissible benchmark** because majority-label assignment would manufacture labels without source/domain authority.

The purpose is to choose a defensible evaluation protocol, not to rank models.



## Reconciliation target

Phase 3A reported a group-aware split of **8,751 / 2,304** rows with F1 ≈ **0.9528**. Phase 3B, using the same canonical file SHA256, reported **8,777 / 2,278** rows with F1 ≈ **0.9564**.

A likely reproducibility issue is that `GroupShuffleSplit` samples *group positions*. If integer group IDs are assigned according to first-seen row order (`pd.factorize(..., sort=False)`), a different group ordering can produce a different valid split even when the underlying groups are identical.

Phase 3C therefore:
1. reconstructs the legacy first-seen factorization;
2. constructs deterministic group IDs based on sorted feature tuples;
3. constructs stable content-hash IDs;
4. compares the resulting group assignments and split indices;
5. uses the deterministic representation for the candidate official group-aware protocol.


In [1]:

from pathlib import Path
import hashlib
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "Result"
EXPECTED_SHA256 = "a4b16abbd8610e4f53fd63e8eb3da793157a961111ed5e9d9d8afa21af866995"

CANDIDATE_PATHS = [
    Path("data/raw/phisingData.csv"),
    Path("notebooks/data/raw/phisingData.csv"),
    Path(r"E:\Projects\Network security log triage agent\data\raw\phisingData.csv"),
    Path(r"E:\Projects\Network security log triage agent\notebooks\data\raw\phisingData.csv"),
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Canonical phisingData.csv was not found in the expected project locations.")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def dataframe_fingerprint(frame):
    payload = pd.util.hash_pandas_object(frame, index=True).values.tobytes()
    return hashlib.sha256(payload).hexdigest()

df = pd.read_csv(DATA_PATH)
actual_sha256 = sha256_file(DATA_PATH)

print("Dataset:", DATA_PATH)
print("Shape:", df.shape)
print("SHA256:", actual_sha256)
print("Target counts:")
print(df[TARGET].value_counts().sort_index())

if actual_sha256.lower() != EXPECTED_SHA256.lower():
    raise ValueError(f"Dataset SHA256 mismatch. Expected {EXPECTED_SHA256}, got {actual_sha256}.")

feature_cols = [c for c in df.columns if c != TARGET]
if len(feature_cols) != 30:
    raise ValueError(f"Expected 30 feature columns, found {len(feature_cols)}.")


Dataset: data\raw\phisingData.csv
Shape: (11055, 31)
SHA256: a4b16abbd8610e4f53fd63e8eb3da793157a961111ed5e9d9d8afa21af866995
Target counts:
Result
-1    4898
 1    6157
Name: count, dtype: int64


In [2]:

dataset_metadata = {
    "path": str(DATA_PATH),
    "sha256": actual_sha256,
    "shape": list(df.shape),
    "dataframe_fingerprint": dataframe_fingerprint(df),
    "target": TARGET,
    "feature_count": len(feature_cols),
    "exact_duplicate_rows": int(df.duplicated(keep=False).sum()),
    "unique_full_rows": int(df.drop_duplicates().shape[0]),
}
dataset_metadata


{'path': 'data\\raw\\phisingData.csv',
 'sha256': 'a4b16abbd8610e4f53fd63e8eb3da793157a961111ed5e9d9d8afa21af866995',
 'shape': [11055, 31],
 'dataframe_fingerprint': '885e849647c229a5a08975c61e39a2e3d94b08ac740a6e975319dbff12703b72',
 'target': 'Result',
 'feature_count': 30,
 'exact_duplicate_rows': 7843,
 'unique_full_rows': 5849}


## 1. Reconstruct feature groups

A feature group is defined by the complete 30-feature vector, excluding `Result`.

Three representations are compared:

- **Legacy first-seen factorization:** `pd.factorize(..., sort=False)`. This mirrors the Phase 3B implementation.
- **Deterministic sorted group IDs:** feature tuples are sorted lexicographically before assigning IDs.
- **Stable content hash IDs:** a SHA256 digest of the feature tuple is used as the group identifier.

All three should encode the same mathematical grouping. Their identifiers are only labels; the key question is whether their ordering changes the `GroupShuffleSplit` sample.


In [3]:

feature_tuples = df[feature_cols].apply(tuple, axis=1)

# Legacy implementation used in Phase 3B.
legacy_groups = pd.Series(
    pd.factorize(feature_tuples, sort=False)[0],
    index=df.index,
    name="legacy_group",
)

# Deterministic lexicographic group IDs.
unique_feature_tuples = sorted(set(feature_tuples.tolist()))
sorted_group_map = {t: i for i, t in enumerate(unique_feature_tuples)}
sorted_groups = feature_tuples.map(sorted_group_map).astype("int64")
sorted_groups.name = "sorted_group"

# Stable content hash identifiers.
def tuple_digest(values):
    return hashlib.sha256(repr(tuple(values)).encode("utf-8")).hexdigest()

hash_groups = feature_tuples.map(tuple_digest)
hash_groups.name = "hash_group"

print("Total feature groups:", legacy_groups.nunique())
print("Legacy/sorted group counts:", legacy_groups.nunique(), sorted_groups.nunique())
print("Stable-hash group count:", hash_groups.nunique())


Total feature groups: 5785
Legacy/sorted group counts: 5785 5785
Stable-hash group count: 5785


In [4]:

def split_indices(groups):
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
    train_idx, test_idx = next(
        splitter.split(df[feature_cols], df[TARGET], groups=groups)
    )
    return np.asarray(train_idx), np.asarray(test_idx)

legacy_train_idx, legacy_test_idx = split_indices(legacy_groups)
sorted_train_idx, sorted_test_idx = split_indices(sorted_groups)
hash_train_idx, hash_test_idx = split_indices(hash_groups)

def index_overlap(a, b):
    return int(len(set(a).intersection(set(b))))

reconciliation = {
    "legacy_vs_sorted": {
        "train_rows_equal": bool(np.array_equal(np.sort(legacy_train_idx), np.sort(sorted_train_idx))),
        "test_rows_equal": bool(np.array_equal(np.sort(legacy_test_idx), np.sort(sorted_test_idx))),
        "train_index_overlap": index_overlap(legacy_train_idx, sorted_train_idx),
        "test_index_overlap": index_overlap(legacy_test_idx, sorted_test_idx),
    },
    "legacy_vs_hash": {
        "train_rows_equal": bool(np.array_equal(np.sort(legacy_train_idx), np.sort(hash_train_idx))),
        "test_rows_equal": bool(np.array_equal(np.sort(legacy_test_idx), np.sort(hash_test_idx))),
        "train_index_overlap": index_overlap(legacy_train_idx, hash_train_idx),
        "test_index_overlap": index_overlap(legacy_test_idx, hash_test_idx),
    },
    "sorted_vs_hash": {
        "train_rows_equal": bool(np.array_equal(np.sort(sorted_train_idx), np.sort(hash_train_idx))),
        "test_rows_equal": bool(np.array_equal(np.sort(sorted_test_idx), np.sort(hash_test_idx))),
        "train_index_overlap": index_overlap(sorted_train_idx, hash_train_idx),
        "test_index_overlap": index_overlap(sorted_test_idx, hash_test_idx),
    },
}
reconciliation


{'legacy_vs_sorted': {'train_rows_equal': False,
  'test_rows_equal': False,
  'train_index_overlap': 6965,
  'test_index_overlap': 446},
 'legacy_vs_hash': {'train_rows_equal': False,
  'test_rows_equal': False,
  'train_index_overlap': 6998,
  'test_index_overlap': 504},
 'sorted_vs_hash': {'train_rows_equal': False,
  'test_rows_equal': False,
  'train_index_overlap': 7029,
  'test_index_overlap': 515}}

In [5]:

def shared_group_count(groups, train_idx, test_idx):
    train_groups = set(pd.Series(groups).iloc[train_idx])
    test_groups = set(pd.Series(groups).iloc[test_idx])
    return int(len(train_groups.intersection(test_groups)))

def evaluate_indices(frame, groups, train_idx, test_idx, protocol_name):
    X = frame[feature_cols].copy()
    y = frame[TARGET].map({-1: 0, 1: 1})
    if y.isna().any():
        raise ValueError("Unexpected target values outside {-1, 1}.")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    imputer = KNNImputer(n_neighbors=3, weights="uniform")
    X_train_t = imputer.fit_transform(X_train)
    X_test_t = imputer.transform(X_test)

    model = RandomForestClassifier(
        n_estimators=128,
        criterion="gini",
        bootstrap=True,
        max_depth=None,
        max_features="sqrt",
        random_state=RANDOM_STATE,
    )
    model.fit(X_train_t, y_train)
    pred = model.predict(X_test_t)

    return {
        "protocol": protocol_name,
        "train_rows": int(len(train_idx)),
        "test_rows": int(len(test_idx)),
        "train_target_counts": {str(k): int(v) for k, v in y_train.value_counts().sort_index().items()},
        "test_target_counts": {str(k): int(v) for k, v in y_test.value_counts().sort_index().items()},
        "shared_feature_groups": shared_group_count(groups, train_idx, test_idx),
        "accuracy": float(accuracy_score(y_test, pred)),
        "f1": float(f1_score(y_test, pred)),
        "precision": float(precision_score(y_test, pred)),
        "recall": float(recall_score(y_test, pred)),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
        "train_index_fingerprint": hashlib.sha256(np.sort(train_idx).astype(np.int64).tobytes()).hexdigest(),
        "test_index_fingerprint": hashlib.sha256(np.sort(test_idx).astype(np.int64).tobytes()).hexdigest(),
    }

legacy_eval = evaluate_indices(
    df, legacy_groups, legacy_train_idx, legacy_test_idx,
    "legacy_group_factorize_sort_false",
)
sorted_eval = evaluate_indices(
    df, sorted_groups, sorted_train_idx, sorted_test_idx,
    "deterministic_sorted_feature_groups",
)
hash_eval = evaluate_indices(
    df, hash_groups, hash_train_idx, hash_test_idx,
    "stable_hash_feature_groups",
)

pd.DataFrame([legacy_eval, sorted_eval, hash_eval])


,protocol,train_rows,test_rows,train_target_counts,test_target_counts,shared_feature_groups,accuracy,f1,precision,recall,confusion_matrix,train_index_fingerprint,test_index_fingerprint
0,legacy_group_factorize_sort_false,8777,2278,"{'0': 3878, '1': 4899}","{'0': 1020, '1': 1258}",0,0.951712,0.956384,0.954114,0.958665,"[[962, 58], [52, 1206]]",0741342b81f91bad2bd14c435e8ebdb78591593913f5bb...,5be63845d39a40d966243f447e5fe46aced77de6510964...
1,deterministic_sorted_feature_groups,8797,2258,"{'0': 3966, '1': 4831}","{'0': 932, '1': 1326}",0,0.948184,0.955463,0.964643,0.946456,"[[886, 46], [71, 1255]]",8ce409452de18f284e44c132f3628b4b929af2a5967b64...,1189855d6296ad48d606e2fa10edc50f24d765ee77d3c3...
2,stable_hash_feature_groups,8772,2283,"{'0': 3902, '1': 4870}","{'0': 996, '1': 1287}",0,0.957074,0.961749,0.966275,0.957265,"[[953, 43], [55, 1232]]",474d539145ee1b72ba8489cf87ae8ed21602b3a2d9c70a...,8468c865dee3066f9f1d600c77d37560faa9d24d7213cf...



### Reconciliation rule

The discrepancy is considered explained if the legacy split differs from the deterministic split while the deterministic sorted and stable-hash representations agree on the same row membership.

That would show that the disagreement is caused by **group-ID ordering interacting with `GroupShuffleSplit`**, rather than a change to the canonical dataset.


In [6]:

tuple_series = df[feature_cols].apply(tuple, axis=1)

def group_relation_is_valid(groups):
    # Every group must map to exactly one complete feature tuple.
    checks = tuple_series.groupby(groups).nunique(dropna=False)
    return bool(checks.max() == 1)

group_equivalence_check = {
    "legacy_valid": group_relation_is_valid(legacy_groups),
    "sorted_valid": group_relation_is_valid(sorted_groups),
    "hash_valid": group_relation_is_valid(hash_groups),
    "legacy_group_count": int(legacy_groups.nunique()),
    "sorted_group_count": int(sorted_groups.nunique()),
    "hash_group_count": int(hash_groups.nunique()),
}
group_equivalence_check


{'legacy_valid': True,
 'sorted_valid': True,
 'hash_valid': True,
 'legacy_group_count': 5785,
 'sorted_group_count': 5785,
 'hash_group_count': 5785}


## 2. Protocol A — random row split

This reproduces the Phase 2 seeded evaluation. It is retained as the historical baseline and makes duplicate overlap measurable.

It is **not leakage-controlled** when duplicate feature groups span train and test.


In [7]:

train_idx_a, test_idx_a = train_test_split(
    np.arange(len(df)),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=None,
)

protocol_a = evaluate_indices(
    df, legacy_groups, train_idx_a, test_idx_a,
    "A_random_row_split",
)

train_groups_a = set(legacy_groups.iloc[train_idx_a])
test_groups_a = set(legacy_groups.iloc[test_idx_a])
test_rows_with_train_duplicate = int(legacy_groups.iloc[test_idx_a].isin(train_groups_a).sum())

protocol_a["shared_feature_groups"] = int(len(train_groups_a.intersection(test_groups_a)))
protocol_a["test_rows_with_train_duplicate"] = test_rows_with_train_duplicate
protocol_a["test_duplicate_overlap_rate"] = float(test_rows_with_train_duplicate / len(test_idx_a))
protocol_a


{'protocol': 'A_random_row_split',
 'train_rows': 8844,
 'test_rows': 2211,
 'train_target_counts': {'0': 3942, '1': 4902},
 'test_target_counts': {'0': 956, '1': 1255},
 'shared_feature_groups': 1143,
 'accuracy': 0.968340117593849,
 'f1': 0.9723320158102767,
 'precision': 0.9647058823529412,
 'recall': 0.9800796812749004,
 'confusion_matrix': [[911, 45], [25, 1230]],
 'train_index_fingerprint': '4f86d11e876c2ef26f0888c1ce690698758915cc3acb0640b0ec4f71631e81fb',
 'test_index_fingerprint': '5ee41c88306eb5585eae19ba3d1a7af603f44fc13f3abef15e3cb039e9ad60da',
 'test_rows_with_train_duplicate': 1447,
 'test_duplicate_overlap_rate': 0.6544549977385798}


## 3. Protocol B — deterministic feature-group-aware split

This is the candidate **official evaluation protocol**.

Identical feature vectors cannot cross the train/test boundary. Group IDs are assigned deterministically from sorted feature tuples, so the split is reproducible independently of first-seen row ordering.

No rows are removed or relabeled.


In [8]:

protocol_b = sorted_eval
protocol_b


{'protocol': 'deterministic_sorted_feature_groups',
 'train_rows': 8797,
 'test_rows': 2258,
 'train_target_counts': {'0': 3966, '1': 4831},
 'test_target_counts': {'0': 932, '1': 1326},
 'shared_feature_groups': 0,
 'accuracy': 0.9481842338352524,
 'f1': 0.9554625047582794,
 'precision': 0.9646425826287471,
 'recall': 0.9464555052790347,
 'confusion_matrix': [[886, 46], [71, 1255]],
 'train_index_fingerprint': '8ce409452de18f284e44c132f3628b4b929af2a5967b64d4da5ab1115a191760',
 'test_index_fingerprint': '1189855d6296ad48d606e2fa10edc50f24d765ee77d3c38969f487d60b59bc35'}


## 4. Protocol C — conflicting groups excluded

This is a **sensitivity analysis**, not a cleaning decision.

A conflicting feature group occurs when its 30-feature vector appears with both `Result=-1` and `Result=1`. Those groups are temporarily excluded to measure their influence on evaluation.

The canonical CSV remains unchanged.


In [9]:

grouped_labels = (
    df.groupby(feature_cols, dropna=False, sort=False)[TARGET]
      .nunique()
)
conflict_tuples = set(grouped_labels[grouped_labels > 1].index.tolist())

conflict_mask = tuple_series.isin(conflict_tuples)
df_consistent = df.loc[~conflict_mask].copy()

consistent_tuples = df_consistent[feature_cols].apply(tuple, axis=1)
consistent_unique = sorted(set(consistent_tuples.tolist()))
consistent_map = {t: i for i, t in enumerate(consistent_unique)}
consistent_groups = consistent_tuples.map(consistent_map).astype("int64")

splitter_c = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)
train_idx_c, test_idx_c = next(
    splitter_c.split(
        df_consistent[feature_cols],
        df_consistent[TARGET],
        groups=consistent_groups,
    )
)

protocol_c = evaluate_indices(
    df_consistent,
    consistent_groups,
    train_idx_c,
    test_idx_c,
    "C_group_aware_conflicting_groups_excluded",
)

protocol_c.update({
    "rows_available": int(len(df_consistent)),
    "rows_excluded_from_canonical": int(len(df) - len(df_consistent)),
    "conflicting_groups": int(len(conflict_tuples)),
    "conflicting_row_rate": float(conflict_mask.mean()),
})
protocol_c


{'protocol': 'C_group_aware_conflicting_groups_excluded',
 'train_rows': 8586,
 'test_rows': 2112,
 'train_target_counts': {'0': 3795, '1': 4791},
 'test_target_counts': {'0': 958, '1': 1154},
 'shared_feature_groups': 0,
 'accuracy': 0.9621212121212122,
 'f1': 0.9652476107732406,
 'precision': 0.9677700348432056,
 'recall': 0.962738301559792,
 'confusion_matrix': [[921, 37], [43, 1111]],
 'train_index_fingerprint': '2f3e46996ca7ada966528e624dc9395150c04a6252d52cb1643bf2828e1938b4',
 'test_index_fingerprint': '5b157cb053a556f5bc70ef224df74a0fc8b7b8731161161837d47a919b64a9a8',
 'rows_available': 10698,
 'rows_excluded_from_canonical': 357,
 'conflicting_groups': 64,
 'conflicting_row_rate': 0.03229308005427409}


## 5. Protocol D — derived-label policy

A deterministic majority-label rule can be computed for a conflicting feature group, but it would replace observed labels with a label derived from the same dataset.

That is **not a neutral evaluation protocol** unless an external source or domain rule establishes that the majority label is authoritative.

Therefore Phase 3C records D as **inadmissible for the official benchmark** rather than manufacturing labels.


In [10]:

protocol_d = {
    "protocol": "D_deterministic_group_majority_label",
    "status": "not_admissible_as_official_benchmark",
    "reason": (
        "Would derive/rewrite labels for conflicting feature groups without "
        "external source or domain authority. This changes ground truth rather "
        "than merely changing the split."
    ),
    "production_dataset_modified": False,
}
protocol_d


{'protocol': 'D_deterministic_group_majority_label',
 'status': 'not_admissible_as_official_benchmark',
 'reason': 'Would derive/rewrite labels for conflicting feature groups without external source or domain authority. This changes ground truth rather than merely changing the split.',
 'production_dataset_modified': False}


## 6. Candidate protocol comparison

This table is descriptive. It is not a model ranking.

For protocol selection, the critical properties are:
- leakage control;
- reproducibility;
- preservation of observed labels;
- transparent conflict handling;
- stability of train/test membership.


In [11]:

comparison = pd.DataFrame([
    {
        "protocol": protocol_a["protocol"],
        "train_rows": protocol_a["train_rows"],
        "test_rows": protocol_a["test_rows"],
        "shared_groups": protocol_a["shared_feature_groups"],
        "duplicate_overlap_rate": protocol_a["test_duplicate_overlap_rate"],
        "f1": protocol_a["f1"],
        "precision": protocol_a["precision"],
        "recall": protocol_a["recall"],
    },
    {
        "protocol": protocol_b["protocol"],
        "train_rows": protocol_b["train_rows"],
        "test_rows": protocol_b["test_rows"],
        "shared_groups": protocol_b["shared_feature_groups"],
        "duplicate_overlap_rate": 0.0,
        "f1": protocol_b["f1"],
        "precision": protocol_b["precision"],
        "recall": protocol_b["recall"],
    },
    {
        "protocol": protocol_c["protocol"],
        "train_rows": protocol_c["train_rows"],
        "test_rows": protocol_c["test_rows"],
        "shared_groups": protocol_c["shared_feature_groups"],
        "duplicate_overlap_rate": 0.0,
        "f1": protocol_c["f1"],
        "precision": protocol_c["precision"],
        "recall": protocol_c["recall"],
    },
])
comparison


,protocol,train_rows,test_rows,shared_groups,duplicate_overlap_rate,f1,precision,recall
0,A_random_row_split,8844,2211,1143,0.654455,0.972332,0.964706,0.980080
1,deterministic_sorted_feature_groups,8797,2258,0,0.000000,0.955463,0.964643,0.946456
2,C_group_aware_conflicting_groups_excluded,8586,2112,0,0.000000,0.965248,0.967770,0.962738



## 7. Decision logic

The proposed evaluation policy is:

1. **Protocol A** remains the historical/random-split baseline.
2. **Protocol B** becomes the primary evaluation protocol only if the deterministic sorted and stable-hash representations agree on the same train/test membership and have zero shared feature groups.
3. **Protocol C** remains sensitivity analysis only.
4. **Protocol D** is not used for official benchmarking without an external authoritative labeling rule.
5. No production code, canonical data, labels, or DVC-tracked files are modified by this experiment.


In [12]:

deterministic_agreement = (
    reconciliation["sorted_vs_hash"]["train_rows_equal"]
    and reconciliation["sorted_vs_hash"]["test_rows_equal"]
    and sorted_eval["shared_feature_groups"] == 0
    and hash_eval["shared_feature_groups"] == 0
)

decision = {
    "official_primary_protocol": (
        "B_deterministic_feature_group_aware_split"
        if deterministic_agreement
        else "UNRESOLVED"
    ),
    "historical_baseline": "A_random_row_split",
    "sensitivity_protocol": "C_group_aware_conflicting_groups_excluded",
    "derived_label_protocol": "D_not_admissible_without_external_label_authority",
    "deterministic_group_methods_agree": bool(deterministic_agreement),
    "production_code_modified": False,
    "canonical_dataset_modified": False,
}
decision


{'official_primary_protocol': 'UNRESOLVED',
 'historical_baseline': 'A_random_row_split',
 'sensitivity_protocol': 'C_group_aware_conflicting_groups_excluded',
 'derived_label_protocol': 'D_not_admissible_without_external_label_authority',
 'deterministic_group_methods_agree': False,
 'production_code_modified': False,
 'canonical_dataset_modified': False}

In [13]:

artifact = {
    "experiment": {
        "name": "phase_3c_evaluation_protocol_comparison",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "status": "completed",
        "production_code_modified": False,
        "canonical_dataset_modified": False,
    },
    "dataset": dataset_metadata,
    "configuration": {
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "target_mapping": {"-1": 0, "1": 1},
        "preprocessing": {
            "transformer": "KNNImputer",
            "n_neighbors": 3,
            "weights": "uniform",
        },
        "model": {
            "type": "RandomForestClassifier",
            "n_estimators": 128,
            "criterion": "gini",
            "bootstrap": True,
            "max_depth": None,
            "max_features": "sqrt",
            "random_state": RANDOM_STATE,
        },
    },
    "reconciliation": reconciliation,
    "group_equivalence": group_equivalence_check,
    "protocols": {
        "A_random_row_split": protocol_a,
        "B_deterministic_feature_group_aware_split": protocol_b,
        "C_conflicting_groups_excluded_sensitivity": protocol_c,
        "D_derived_label_policy": protocol_d,
    },
    "decision": decision,
    "notes": [
        "Protocol A allows feature-group overlap and is retained only as the historical random-split baseline.",
        "Protocol B is primary only when deterministic group representations agree and shared groups are zero.",
        "Protocol C is sensitivity analysis and does not justify deletion or relabeling of canonical rows.",
        "Protocol D is not an official benchmark because it derives labels without external authority.",
    ],
}

ARTIFACT_PATH = Path("evaluation/phase_3c_evaluation_protocol_comparison.json")
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)

with ARTIFACT_PATH.open("w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2)

print(f"Artifact written to: {ARTIFACT_PATH.resolve()}")

# artifact_path = Path("evaluation/phase_3c_evaluation_protocol_comparison.json")
# artifact_path.parent.mkdir(parents=True, exist_ok=True)
# artifact_path.write_text(json.dumps(artifact, indent=2), encoding="utf-8")
# print("Wrote:", artifact_path)
# print(json.dumps(decision, indent=2))


Artifact written to: E:\Projects\Network security log triage agent\notebooks\evaluation\phase_3c_evaluation_protocol_comparison.json
